# Jane Street Real-Time Market Forecasting with `dv_jax_ml` + JAX

This notebook is a faithful JAX implementation of the main ideas Patrick Yams
to "Jane Street Real-Time Market Forecasting" Competition. Unforunately there was not documenatation of his exact set up so this is a recreation based on my own infrence of how the model should be structured, based on Jax and functions from my own library.



In [26]:
!pip install git+https://github.com/dvat-oss-source/dv_jax_ml_library.git



  Cloning https://github.com/dvat-oss-source/dv_jax_ml_library.git to /tmp/pip-req-build-4jxt6hmn
  Running command git clone --filter=blob:none --quiet https://github.com/dvat-oss-source/dv_jax_ml_library.git /tmp/pip-req-build-4jxt6hmn
  Resolved https://github.com/dvat-oss-source/dv_jax_ml_library.git to commit b305862e8453651e7c17cfd65d942ce55e7c2c27
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [27]:
import os
from pathlib import Path

print("Available inputs in /kaggle/input:")
print(os.listdir("/kaggle/input"))

Available inputs in /kaggle/input:
['competitions']


In [28]:
import os
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

from dataclasses import dataclass
from functools import partial
from pathlib import Path
import pickle
import subprocess
import sys
from typing import Any, Iterator

try:
    from dv_jax_ml import Adam, LayerNorm, Linear
except ImportError:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "git+https://github.com/dvat-oss-source/dv_jax_ml_library.git",
        ]
    )
    from dv_jax_ml import Adam, LayerNorm, Linear

import jax
import jax.numpy as jnp
import jax.scipy as jsp
import numpy as np
import polars as pl

jax.config.update("jax_enable_x64", False)
# Kaggle GPU T4 x2 exposes both cards as local JAX devices.
_gpu_devices = tuple(jax.local_devices(backend="gpu")) if any(
    device.platform == "gpu" for device in jax.local_devices()
) else ()
LOCAL_DEVICES = _gpu_devices or tuple(jax.local_devices())
N_LOCAL_DEVICES = len(LOCAL_DEVICES)
print("JAX local devices:", LOCAL_DEVICES)
if N_LOCAL_DEVICES < 2:
    print("Only one device is visible. In Kaggle, select Accelerator: GPU T4 x2 to use both T4s.")

JAX devices: [CpuDevice(id=0)]


## Configuration

`max_time_steps=968` follows the full-day sequence length reported in the
solution talk. `extra_symbol_slots` permits a few unseen symbols at inference
while sharing one unknown-symbol embedding. Use a GPU Kaggle notebook and
increase `seeds`, `offline_epochs`, and `max_days_per_epoch` only after the
smoke test succeeds.

In [29]:
@dataclass(frozen=True)
class TrainConfig:
    data_root: str = "/kaggle/input/competitions/jane-street-real-time-market-data-forecasting"    
    max_time_steps: int = 968
    extra_symbol_slots: int = 16
    d_model: int = 64
    gru_width_multiplier: int = 2
    num_heads: int = 4
    num_axial_blocks: int = 2
    ffn_multiplier: int = 2
    feature_clip: float = 5.0
    learning_rate: float = 2e-4
    weight_decay: float = 1e-5
    offline_epochs: int = 1
    max_days_per_epoch: int | None = None
    seeds: tuple[int, ...] = (13, 29, 47, 61)
    online_learning_rate: float = 5e-5
    online_steps: int = 3
    beta1: float = 0.85
    beta2: float = 0.98
    gradient_clip_norm: float = 1.0
    recency_offset: float = 200.0
    full_sequence_multiplier: float = 1.5


CFG = TrainConfig()
FEATURE_COLS = [f"feature_{index:02d}" for index in range(79)]
TARGET_COLS = [f"responder_{index}" for index in range(9)]
REQUIRED_TRAIN_COLS = ["date_id", "time_id", "symbol_id", "weight", *FEATURE_COLS, *TARGET_COLS]

## Read metadata and fit preprocessing on training data only

The model sees the original 79 numerical features plus a time-of-day feature.
Time is mapped from its globally scaled rank to a Gaussian quantile. Numerical
means/stds are global, as in the talk; no future test value is used here.

In [ ]:
def parquet_files(path: Path) -> list[str]:
    if path.is_file():
        return [str(path)]
    files = sorted(str(file) for file in path.rglob("*.parquet"))
    if not files:
        raise FileNotFoundError(f"No parquet files found below {path}")
    return files


@dataclass(frozen=True)
class FeatureScaler:
    mean: np.ndarray
    std: np.ndarray


@dataclass(frozen=True)
class Metadata:
    symbol_ids: np.ndarray
    time_ids: np.ndarray
    date_min: int
    date_max: int
    time_min: int
    time_max: int
    max_symbols: int

    @property
    def unknown_token(self) -> int:
        return len(self.symbol_ids)

    @property
    def num_symbol_embeddings(self) -> int:
        return len(self.symbol_ids) + 1

    @property
    def time_to_slot(self) -> dict[int, int]:
        return {int(time_id): index for index, time_id in enumerate(self.time_ids)}


def fit_metadata_and_scaler(files: list[str], cfg: TrainConfig) -> tuple[Metadata, FeatureScaler]:
    scan = pl.scan_parquet(files)
    identifiers = scan.select(["date_id", "time_id", "symbol_id"]).collect(engine="streaming")
    symbol_ids = np.sort(identifiers.get_column("symbol_id").unique().to_numpy().astype(np.int32))
    time_ids = np.sort(identifiers.get_column("time_id").unique().to_numpy().astype(np.int32))
    date_ids = identifiers.get_column("date_id").to_numpy().astype(np.int32)

    if len(time_ids) > cfg.max_time_steps:
        raise ValueError(
            f"Found {len(time_ids)} distinct time ids, above max_time_steps={cfg.max_time_steps}. "
            "Increase the configuration before compiling the model."
        )
    if len(symbol_ids) + cfg.extra_symbol_slots > 512:
        raise ValueError("Too many symbols for the default tensor layout; revise the symbol-slot strategy.")

    stat_expressions = []
    for column in FEATURE_COLS:
        clean = pl.col(column).cast(pl.Float32).fill_nan(None)
        stat_expressions.extend([clean.mean().alias(f"{column}_mean"), clean.std().alias(f"{column}_std")])
    stats = scan.select(stat_expressions).collect(engine="streaming")
    mean = np.asarray([stats[f"{column}_mean"][0] for column in FEATURE_COLS], dtype=np.float32)
    std = np.asarray([stats[f"{column}_std"][0] for column in FEATURE_COLS], dtype=np.float32)
    mean = np.nan_to_num(mean, nan=0.0, posinf=0.0, neginf=0.0)
    std = np.where(np.isfinite(std) & (std > 1e-6), std, 1.0).astype(np.float32)

    metadata = Metadata(
        symbol_ids=symbol_ids,
        time_ids=time_ids,
        date_min=int(date_ids.min()),
        date_max=int(date_ids.max()),
        time_min=int(time_ids.min()),
        time_max=int(time_ids.max()),
        max_symbols=len(symbol_ids) + cfg.extra_symbol_slots,
    )
    return metadata, FeatureScaler(mean=mean, std=std)

def parquet_files(path: Path) -> list[str]:
    path = path.resolve()
    if not path.exists():
        raise FileNotFoundError(f"Path does not exist: {path}")
    if path.is_file():
        return [str(path)]
    
    # Recursively find all parquet files in partition subdirectories
    files = sorted(str(f) for f in path.glob("**/*.parquet") if f.is_file())
    if not files:
        files = sorted(str(f) for f in path.rglob("*") if f.is_file())
    if not files:
        raise FileNotFoundError(f"No parquet files found below {path}")
    return files

DATA_ROOT = Path(CFG.data_root)
# Point directly to the train dataset directory
TRAIN_FILES = parquet_files(DATA_ROOT / "train.parquet")
METADATA, SCALER = fit_metadata_and_scaler(TRAIN_FILES, CFG)
print(f"Training files: {len(TRAIN_FILES)}")
print(f"Known symbols: {len(METADATA.symbol_ids)}, tensor slots: {METADATA.max_symbols}")
print(f"Known time ids: {len(METADATA.time_ids)}, padded length: {CFG.max_time_steps}")


def estimate_target_weights(files: list[str], sample_rows: int = 500_000) -> np.ndarray:
    responders = pl.scan_parquet(files).select(TARGET_COLS).collect(engine="streaming").to_numpy().astype(np.float32)
    responders = responders[np.all(np.isfinite(responders), axis=1)]
    if len(responders) > sample_rows:
        chooser = np.random.default_rng(123).choice(len(responders), size=sample_rows, replace=False)
        responders = responders[chooser]
    correlation = np.corrcoef(responders, rowvar=False)
    weights = np.abs(correlation[:, 6])
    weights = np.maximum(weights / weights[6], 0.05)
    weights[6] = 1.0
    print("Target correlation with responder_6:", np.round(correlation[:, 6], 3))
    print("Multi-task target weights:", np.round(weights, 3))
    return weights.astype(np.float32)


TARGET_WEIGHTS = jnp.asarray(estimate_target_weights(TRAIN_FILES))

## Fixed-shape day tensors

A fixed `[time, symbol, feature]` layout avoids recompilation. Symbol slots are
stable for every training symbol. At inference, spare slots hold new symbols
for the duration of the current day, while all such symbols share the unknown
embedding. The `valid` and `target_mask` arrays ensure padding never affects
attention, recurrent state, loss, or score.

In [ ]:
NUM_NUMERIC_FEATURES = len(FEATURE_COLS) + 1


def normalise_features(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=np.float32)
    clean = np.where(np.isfinite(values), values, SCALER.mean)
    return np.clip((clean - SCALER.mean) / SCALER.std, -CFG.feature_clip, CFG.feature_clip).astype(np.float32)


def gaussian_time_feature(time_ids: np.ndarray) -> np.ndarray:
    denominator = max(METADATA.time_max - METADATA.time_min, 1)
    uniform = (np.asarray(time_ids, dtype=np.float32) - METADATA.time_min) / denominator
    uniform = np.clip(uniform, 1e-4, 1.0 - 1e-4)
    return np.asarray(jsp.special.ndtri(jnp.asarray(uniform)), dtype=np.float32)


def known_symbol_slots(symbol_ids: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    symbol_ids = np.asarray(symbol_ids, dtype=np.int32)
    insertion = np.searchsorted(METADATA.symbol_ids, symbol_ids)
    known = (insertion < len(METADATA.symbol_ids)) & (METADATA.symbol_ids[np.minimum(insertion, len(METADATA.symbol_ids) - 1)] == symbol_ids)
    if not np.all(known):
        unknown = np.unique(symbol_ids[~known])
        raise ValueError(f"Training data contains symbols absent from fitted metadata: {unknown[:5]}")
    return insertion.astype(np.int32), insertion.astype(np.int32)


def empty_day_batch() -> dict[str, np.ndarray]:
    shape = (CFG.max_time_steps, METADATA.max_symbols)
    return {
        "features": np.zeros((*shape, NUM_NUMERIC_FEATURES), dtype=np.float32),
        "tokens": np.full(shape, METADATA.unknown_token, dtype=np.int32),
        "valid": np.zeros(shape, dtype=bool),
        "target": np.zeros((*shape, len(TARGET_COLS)), dtype=np.float32),
        "target_mask": np.zeros(shape, dtype=bool),
        "weights": np.zeros(shape, dtype=np.float32),
    }


def make_training_day(day: pl.DataFrame) -> tuple[int, dict[str, np.ndarray]]:
    day = day.sort(["time_id", "symbol_id"])
    date_id = int(day["date_id"][0])
    time_ids = day["time_id"].to_numpy().astype(np.int32)
    symbol_ids = day["symbol_id"].to_numpy().astype(np.int32)
    time_slots = np.fromiter((METADATA.time_to_slot[int(time_id)] for time_id in time_ids), dtype=np.int32)
    symbol_slots, tokens = known_symbol_slots(symbol_ids)
    batch = empty_day_batch()

    numeric = normalise_features(day.select(FEATURE_COLS).to_numpy())
    numeric = np.concatenate([numeric, gaussian_time_feature(time_ids)[:, None]], axis=1)
    targets = day.select(TARGET_COLS).to_numpy().astype(np.float32)
    targets = np.nan_to_num(targets, nan=0.0, posinf=5.0, neginf=-5.0)
    raw_weight = np.nan_to_num(day["weight"].to_numpy().astype(np.float32), nan=0.0)

    recency = (date_id - METADATA.date_min + CFG.recency_offset) / (
        METADATA.date_max - METADATA.date_min + CFG.recency_offset
    )
    is_full_sequence = len(np.unique(time_ids)) == len(METADATA.time_ids)
    sequence_weight = CFG.full_sequence_multiplier if is_full_sequence else 1.0

    batch["features"][time_slots, symbol_slots] = numeric
    batch["tokens"][time_slots, symbol_slots] = tokens
    batch["valid"][time_slots, symbol_slots] = True
    batch["target"][time_slots, symbol_slots] = targets
    batch["target_mask"][time_slots, symbol_slots] = True
    batch["weights"][time_slots, symbol_slots] = raw_weight * recency * sequence_weight
    return date_id, batch


def iter_training_days(files: list[str], seed: int) -> Iterator[tuple[int, dict[str, np.ndarray]]]:
    generator = np.random.default_rng(seed)
    paths = np.asarray(files, dtype=object)
    for path in generator.permutation(paths):
        frame = pl.read_parquet(str(path), columns=REQUIRED_TRAIN_COLS).sort(["date_id", "time_id", "symbol_id"])
        for day in frame.partition_by("date_id", maintain_order=True):
            yield make_training_day(day)

## Stateless axial model

There are no mutable layer objects in the compiled path. `params` is a plain
pytree, and the GRU carry is an explicit input/output. `Linear` and `LayerNorm`
initialise parameter layouts compatible with `dv_jax_ml`; the pure functions
below apply those pytrees inside `jax.jit` without capturing state.

In [ ]:
@dataclass(frozen=True)
class ModelConfig:
    num_features: int
    num_symbols: int
    max_symbols: int
    d_model: int
    gru_hidden: int
    num_heads: int
    num_blocks: int
    ffn_dim: int
    num_targets: int


MODEL_CFG = ModelConfig(
    num_features=NUM_NUMERIC_FEATURES,
    num_symbols=METADATA.num_symbol_embeddings,
    max_symbols=METADATA.max_symbols,
    d_model=CFG.d_model,
    gru_hidden=CFG.d_model * CFG.gru_width_multiplier,
    num_heads=CFG.num_heads,
    num_blocks=CFG.num_axial_blocks,
    ffn_dim=CFG.d_model * CFG.ffn_multiplier,
    num_targets=len(TARGET_COLS),
)
assert MODEL_CFG.d_model % MODEL_CFG.num_heads == 0


def init_linear(key: jax.Array, input_size: int, output_size: int) -> dict[str, jax.Array]:
    return Linear(input_size, output_size).init(key)


def linear(params: dict[str, jax.Array], values: jax.Array) -> jax.Array:
    return values @ params["weights"] + params["bias"]


def init_norm(width: int) -> dict[str, jax.Array]:
    return LayerNorm(width).init()


def layer_norm(params: dict[str, jax.Array], values: jax.Array, eps: float = 1e-5) -> jax.Array:
    mean = jnp.mean(values, axis=-1, keepdims=True)
    variance = jnp.mean(jnp.square(values - mean), axis=-1, keepdims=True)
    return params["gamma"] * (values - mean) * jax.lax.rsqrt(variance + eps) + params["beta"]


def init_cross_attention(key: jax.Array) -> dict[str, Any]:
    keys = jax.random.split(key, 4)
    return {name: init_linear(subkey, MODEL_CFG.d_model, MODEL_CFG.d_model) for name, subkey in zip(("q", "k", "v", "out"), keys)}


def cross_symbol_attention(params: dict[str, Any], values: jax.Array, valid: jax.Array) -> jax.Array:
    head_dim = MODEL_CFG.d_model // MODEL_CFG.num_heads
    q = linear(params["q"], values).reshape(MODEL_CFG.max_symbols, MODEL_CFG.num_heads, head_dim)
    k = linear(params["k"], values).reshape(MODEL_CFG.max_symbols, MODEL_CFG.num_heads, head_dim)
    v = linear(params["v"], values).reshape(MODEL_CFG.max_symbols, MODEL_CFG.num_heads, head_dim)
    scores = jnp.einsum("shd,khd->hsk", q, k) / jnp.sqrt(jnp.asarray(head_dim, dtype=values.dtype))
    scores = jnp.where(valid[None, None, :], scores, -1e9)
    attention = jax.nn.softmax(scores, axis=-1)
    attention = jnp.where(valid[None, :, None], attention, 0.0)
    mixed = jnp.einsum("hsk,khd->shd", attention, v).reshape(MODEL_CFG.max_symbols, MODEL_CFG.d_model)
    return linear(params["out"], mixed)


def init_gru(key: jax.Array) -> dict[str, jax.Array]:
    key_x, key_h = jax.random.split(key)
    scale_x = jnp.sqrt(1.0 / MODEL_CFG.d_model)
    scale_h = jnp.sqrt(1.0 / MODEL_CFG.gru_hidden)
    return {
        "w_x": jax.random.normal(key_x, (MODEL_CFG.d_model, 3 * MODEL_CFG.gru_hidden)) * scale_x,
        "w_h": jax.random.normal(key_h, (MODEL_CFG.gru_hidden, 3 * MODEL_CFG.gru_hidden)) * scale_h,
        "bias": jnp.zeros((3 * MODEL_CFG.gru_hidden,), dtype=jnp.float32),
    }


def gru_step(params: dict[str, jax.Array], values: jax.Array, valid: jax.Array, hidden: jax.Array) -> jax.Array:
    x_z, x_r, x_n = jnp.split(values @ params["w_x"] + params["bias"], 3, axis=-1)
    h_z, h_r, h_n = jnp.split(hidden @ params["w_h"], 3, axis=-1)
    update = jax.nn.sigmoid(x_z + h_z)
    reset = jax.nn.sigmoid(x_r + h_r)
    candidate = jnp.tanh(x_n + reset * h_n)
    new_hidden = (1.0 - update) * hidden + update * candidate
    return jnp.where(valid[:, None], new_hidden, hidden)


def init_axial_block(key: jax.Array) -> dict[str, Any]:
    keys = jax.random.split(key, 8)
    return {
        "cross_norm": init_norm(MODEL_CFG.d_model),
        "cross": init_cross_attention(keys[0]),
        "time_norm": init_norm(MODEL_CFG.d_model),
        "gru": init_gru(keys[1]),
        "time_out": init_linear(keys[2], MODEL_CFG.gru_hidden, MODEL_CFG.d_model),
        "ffn_norm": init_norm(MODEL_CFG.d_model),
        "ffn_in": init_linear(keys[3], MODEL_CFG.d_model, MODEL_CFG.ffn_dim),
        "ffn_out": init_linear(keys[4], MODEL_CFG.ffn_dim, MODEL_CFG.d_model),
    }


def init_model(key: jax.Array) -> dict[str, Any]:
    keys = jax.random.split(key, MODEL_CFG.num_blocks + 3)
    return {
        "feature_proj": init_linear(keys[0], MODEL_CFG.num_features, MODEL_CFG.d_model),
        "symbol_embedding": jax.random.normal(keys[1], (MODEL_CFG.num_symbols, MODEL_CFG.d_model)) * 0.02,
        "blocks": [init_axial_block(block_key) for block_key in keys[2:-1]],
        "head_norm": init_norm(MODEL_CFG.d_model),
        "head": init_linear(keys[-1], MODEL_CFG.d_model, MODEL_CFG.num_targets),
    }


def init_carry() -> tuple[jax.Array, ...]:
    return tuple(
        jnp.zeros((MODEL_CFG.max_symbols, MODEL_CFG.gru_hidden), dtype=jnp.float32)
        for _ in range(MODEL_CFG.num_blocks)
    )


def axial_block_step(
    params: dict[str, Any], values: jax.Array, valid: jax.Array, hidden: jax.Array
) -> tuple[jax.Array, jax.Array]:
    values = values + cross_symbol_attention(params["cross"], layer_norm(params["cross_norm"], values), valid)
    hidden = gru_step(params["gru"], layer_norm(params["time_norm"], values), valid, hidden)
    values = values + linear(params["time_out"], hidden)
    feed_forward = linear(params["ffn_out"], jax.nn.gelu(linear(params["ffn_in"], layer_norm(params["ffn_norm"], values))))
    values = values + feed_forward
    return jnp.where(valid[:, None], values, 0.0), hidden


def model_apply_step(
    params: dict[str, Any], step: dict[str, jax.Array], carries: tuple[jax.Array, ...]
) -> tuple[jax.Array, tuple[jax.Array, ...]]:
    values = linear(params["feature_proj"], jnp.clip(step["features"], -CFG.feature_clip, CFG.feature_clip))
    values = values + params["symbol_embedding"][step["tokens"]]
    next_carries = []
    for block, hidden in zip(params["blocks"], carries):
        values, next_hidden = axial_block_step(block, values, step["valid"], hidden)
        next_carries.append(next_hidden)
    prediction = linear(params["head"], layer_norm(params["head_norm"], values))
    return jnp.where(step["valid"][:, None], prediction, 0.0), tuple(next_carries)


def model_apply_day(
    params: dict[str, Any], batch: dict[str, jax.Array], carries: tuple[jax.Array, ...] | None = None
) -> tuple[jax.Array, tuple[jax.Array, ...]]:
    carries = init_carry() if carries is None else carries

    def scan_step(current_carries: tuple[jax.Array, ...], step: dict[str, jax.Array]):
        prediction, next_carries = model_apply_step(params, step, current_carries)
        return next_carries, prediction

    final_carries, predictions = jax.lax.scan(scan_step, carries, batch)
    return predictions, final_carries

## Weighted multitask zero-mean R² and fully compiled updates

Every responder has its own weighted zero-mean R² loss. Dividing it by a
stop-gradient copy equalises loss scales without changing its local gradient
direction, matching the approximation explained in the talk. Offline and
online optimizers are separate; online adaptation starts a fresh Adam state
for each newly revealed lagged day.

In [ ]:
def weighted_multitask_r2_loss(params: dict[str, Any], batch: dict[str, jax.Array]) -> jax.Array:
    predictions, _ = model_apply_day(params, batch)
    mask = batch["target_mask"][..., None].astype(jnp.float32)
    weights = batch["weights"][..., None] * mask
    residual = jnp.sum(weights * jnp.square(predictions - batch["target"]), axis=(0, 1))
    baseline = jnp.sum(weights * jnp.square(batch["target"]), axis=(0, 1))
    per_target_loss = residual / (baseline + 1e-6)
    scale = jax.lax.stop_gradient(jnp.maximum(per_target_loss, 1e-4))
    return jnp.sum(TARGET_WEIGHTS * (per_target_loss / scale)) / jnp.sum(TARGET_WEIGHTS)


def clip_grads(grads: Any) -> Any:
    squared_norm = sum(jnp.sum(jnp.square(leaf)) for leaf in jax.tree_util.tree_leaves(grads))
    scale = jnp.minimum(1.0, CFG.gradient_clip_norm / (jnp.sqrt(squared_norm) + 1e-6))
    return jax.tree_util.tree_map(lambda gradient: gradient * scale, grads)


OFFLINE_ADAM = Adam(
    lr=CFG.learning_rate,
    b1=CFG.beta1,
    b2=CFG.beta2,
    weight_decay=CFG.weight_decay,
)
ONLINE_ADAM = Adam(
    lr=CFG.online_learning_rate,
    b1=CFG.beta1,
    b2=CFG.beta2,
    weight_decay=CFG.weight_decay,
)


def make_train_step(optimizer: Adam):
    @jax.jit
    def train_step(params: dict[str, Any], optimizer_state: dict[str, Any], batch: dict[str, jax.Array]):
        loss, grads = jax.value_and_grad(weighted_multitask_r2_loss)(params, batch)
        params, optimizer_state = optimizer.update(params, clip_grads(grads), optimizer_state)
        return params, optimizer_state, loss

    return train_step


def make_online_update(optimizer: Adam):
    @jax.jit
    def online_update(params: dict[str, Any], batch: dict[str, jax.Array]):
        optimizer_state = optimizer.init(params)

        def update_once(_: int, state: tuple[dict[str, Any], dict[str, Any], jax.Array]):
            current_params, current_optimizer_state, _ = state
            loss, grads = jax.value_and_grad(weighted_multitask_r2_loss)(current_params, batch)
            next_params, next_optimizer_state = optimizer.update(
                current_params, clip_grads(grads), current_optimizer_state
            )
            return next_params, next_optimizer_state, loss

        params, _, loss = jax.lax.fori_loop(0, CFG.online_steps, update_once, (params, optimizer_state, jnp.inf))
        return params, loss

    return online_update


TRAIN_STEP = make_train_step(OFFLINE_ADAM)
ONLINE_UPDATE = make_online_update(ONLINE_ADAM)


@jax.jit
def ensemble_predict_step(
    parameter_stack: dict[str, Any], step: dict[str, jax.Array], carry_stack: tuple[jax.Array, ...]
) -> tuple[jax.Array, tuple[jax.Array, ...]]:
    predictions, next_carries = jax.vmap(model_apply_step, in_axes=(0, None, 0))(parameter_stack, step, carry_stack)
    return jnp.mean(predictions, axis=0), next_carries


def stack_trees(trees: list[Any]) -> Any:
    return jax.tree_util.tree_map(lambda *leaves: jnp.stack(leaves), *trees)


def to_jax_batch(batch: dict[str, np.ndarray]) -> dict[str, jax.Array]:
    return {name: jnp.asarray(value) for name, value in batch.items()}


smoke_batch = to_jax_batch(empty_day_batch())
smoke_params = init_model(jax.random.key(0))
smoke_state = OFFLINE_ADAM.init(smoke_params)
_, _, smoke_loss = TRAIN_STEP(smoke_params, smoke_state, smoke_batch)
print("Compiled stateless training step; smoke loss:", float(smoke_loss))

## Offline seed ensemble

The four seeds are distributed with `pmap`: on Kaggle T4 x2, each GPU trains
two independent models on the same day. If only one device is visible, all
four models run on that device. Keep the seed count divisible by the visible
device count.


In [ ]:
if len(CFG.seeds) % N_LOCAL_DEVICES:
    raise ValueError(
        f"Number of seeds ({len(CFG.seeds)}) must be divisible by visible devices ({N_LOCAL_DEVICES})."
    )
MODELS_PER_DEVICE = len(CFG.seeds) // N_LOCAL_DEVICES
print(f"Training {len(CFG.seeds)} models across {N_LOCAL_DEVICES} device(s), {MODELS_PER_DEVICE} per device")


def shard_model_axis(parameter_stack: dict[str, Any]) -> dict[str, Any]:
    host_parameters = jax.device_get(parameter_stack)
    shards = []
    for device_index in range(N_LOCAL_DEVICES):
        start = device_index * MODELS_PER_DEVICE
        stop = start + MODELS_PER_DEVICE
        # Bind the loop bounds now; otherwise Python closes over their final values.
        shards.append(
            jax.tree_util.tree_map(
                lambda leaf, start=start, stop=stop: leaf[start:stop], host_parameters
            )
        )
    return jax.device_put_sharded(shards, LOCAL_DEVICES)


def replicated_batch(batch: dict[str, jax.Array]) -> dict[str, np.ndarray]:
    return jax.tree_util.tree_map(
        lambda leaf: np.broadcast_to(np.asarray(leaf), (N_LOCAL_DEVICES, *leaf.shape)).copy(), batch
    )


def init_distributed_optimizer(params: dict[str, Any]) -> dict[str, Any]:
    state = OFFLINE_ADAM.init(params)
    # Adam initializes `step` as scalar Python 0. `pmap` consumes the device
    # axis and nested `vmap` consumes the model axis, so the counter needs
    # both axes just like Adam's m/v state leaves.
    return {**state, "step": jnp.zeros((N_LOCAL_DEVICES, MODELS_PER_DEVICE), dtype=jnp.int32)}


def distributed_train_step(params: dict[str, Any], optimizer_state: dict[str, Any], batch: dict[str, jax.Array]):
    return jax.vmap(TRAIN_STEP, in_axes=(0, 0, None))(params, optimizer_state, batch)


def distributed_online_update(params: dict[str, Any], batch: dict[str, jax.Array]):
    return jax.vmap(ONLINE_UPDATE, in_axes=(0, None))(params, batch)


def distributed_predict_step(params: dict[str, Any], step: dict[str, jax.Array], carries: tuple[jax.Array, ...]):
    predictions, next_carries = jax.vmap(model_apply_step, in_axes=(0, None, 0))(params, step, carries)
    total = jax.lax.psum(jnp.sum(predictions, axis=0), axis_name="ensemble_devices")
    count = jax.lax.psum(jnp.asarray(predictions.shape[0], predictions.dtype), axis_name="ensemble_devices")
    return total / count, next_carries


PMAP_TRAIN_STEP = jax.pmap(distributed_train_step, axis_name="ensemble_devices", devices=LOCAL_DEVICES)
PMAP_ONLINE_UPDATE = jax.pmap(distributed_online_update, axis_name="ensemble_devices", devices=LOCAL_DEVICES)
PMAP_PREDICT_STEP = jax.pmap(distributed_predict_step, axis_name="ensemble_devices", devices=LOCAL_DEVICES)


def train_distributed_ensemble(seeds: tuple[int, ...]) -> dict[str, Any]:
    params = shard_model_axis(stack_trees([init_model(jax.random.key(seed)) for seed in seeds]))
    optimizer_state = init_distributed_optimizer(params)
    for epoch in range(CFG.offline_epochs):
        losses = []
        for day_number, (_, numpy_batch) in enumerate(iter_training_days(TRAIN_FILES, seeds[0] + epoch)):
            params, optimizer_state, loss = PMAP_TRAIN_STEP(
                params, optimizer_state, replicated_batch(to_jax_batch(numpy_batch))
            )
            losses.append(float(np.asarray(loss).mean()))
            if CFG.max_days_per_epoch is not None and day_number + 1 >= CFG.max_days_per_epoch:
                break
        print(f"epoch={epoch + 1}/{CFG.offline_epochs} mean_loss={np.mean(losses):.6f} days={len(losses)} devices={N_LOCAL_DEVICES}")
    return params


PARAMETER_SHARDS = train_distributed_ensemble(CFG.seeds)
MODEL_PATH = Path("/kaggle/working/jane_street_dv_jax_ml_params.pkl")
with MODEL_PATH.open("wb") as handle:
    pickle.dump(jax.device_get(PARAMETER_SHARDS), handle)
print(f"Saved {len(CFG.seeds)} models sharded over {N_LOCAL_DEVICES} device(s) to {MODEL_PATH}")

## Competition inference and daily online adaptation

The API exposes `lags` at the first timestep of the next day. This wrapper
caches the previous day's features on the host, aligns revealed lagged targets
to those features, runs a fresh three-step online Adam update per seed, and
resets the within-day GRU carry. Each individual timestep uses the compiled
`pmap` across all visible devices, so it never recomputes the day's history.

In [ ]:
def empty_runtime_day() -> dict[str, Any]:
    cache = empty_day_batch()
    cache["slot_by_unknown_symbol"] = {}
    return cache


def infer_slots(symbol_ids: np.ndarray, cache: dict[str, Any]) -> tuple[np.ndarray, np.ndarray]:
    symbol_ids = np.asarray(symbol_ids, dtype=np.int32)
    insertion = np.searchsorted(METADATA.symbol_ids, symbol_ids)
    in_bounds = insertion < len(METADATA.symbol_ids)
    known = in_bounds & (METADATA.symbol_ids[np.minimum(insertion, len(METADATA.symbol_ids) - 1)] == symbol_ids)
    slots = insertion.astype(np.int32)
    tokens = np.where(known, insertion, METADATA.unknown_token).astype(np.int32)
    for row_index in np.flatnonzero(~known):
        symbol_id = int(symbol_ids[row_index])
        if symbol_id not in cache["slot_by_unknown_symbol"]:
            next_slot = len(METADATA.symbol_ids) + len(cache["slot_by_unknown_symbol"])
            if next_slot >= METADATA.max_symbols:
                raise ValueError("Unseen-symbol capacity exhausted; increase extra_symbol_slots.")
            cache["slot_by_unknown_symbol"][symbol_id] = next_slot
        slots[row_index] = cache["slot_by_unknown_symbol"][symbol_id]
    return slots, tokens


def prepare_inference_step(test: pl.DataFrame, cache: dict[str, Any]) -> tuple[dict[str, jax.Array], np.ndarray, int]:
    time_values = test["time_id"].to_numpy().astype(np.int32)
    if len(np.unique(time_values)) != 1:
        raise ValueError("The competition API must supply exactly one time_id per predict call.")
    time_id = int(time_values[0])
    if time_id not in METADATA.time_to_slot:
        raise ValueError(f"Unrecognised time_id {time_id}; update the padded time configuration.")
    time_slot = METADATA.time_to_slot[time_id]
    symbol_ids = test["symbol_id"].to_numpy().astype(np.int32)
    slots, tokens = infer_slots(symbol_ids, cache)
    if len(np.unique(slots)) != len(slots):
        raise ValueError("Duplicate symbols in one timestep are not supported by the tensor representation.")

    numeric = normalise_features(test.select(FEATURE_COLS).to_numpy())
    numeric = np.concatenate([numeric, gaussian_time_feature(time_values)[:, None]], axis=1)
    raw_weight = np.nan_to_num(test["weight"].to_numpy().astype(np.float32), nan=0.0)
    cache["features"][time_slot, slots] = numeric
    cache["tokens"][time_slot, slots] = tokens
    cache["valid"][time_slot, slots] = True
    cache["weights"][time_slot, slots] = raw_weight

    step = {
        "features": jnp.asarray(cache["features"][time_slot]),
        "tokens": jnp.asarray(cache["tokens"][time_slot]),
        "valid": jnp.asarray(cache["valid"][time_slot]),
    }
    return step, slots, time_slot


def lag_target_columns(lags: pl.DataFrame) -> list[str]:
    names = [f"responder_{index}_lag_1" for index in range(9)]
    if all(name in lags.columns for name in names):
        return names
    if all(name in lags.columns for name in TARGET_COLS):
        return TARGET_COLS
    raise KeyError("Could not find lagged responder columns in the API payload.")


def batch_with_lagged_targets(lags: pl.DataFrame, cache: dict[str, Any]) -> dict[str, jax.Array] | None:
    if lags is None or lags.height == 0:
        return None
    target_columns = lag_target_columns(lags)
    time_values = lags["time_id"].to_numpy().astype(np.int32)
    symbol_values = lags["symbol_id"].to_numpy().astype(np.int32)
    target_values = np.nan_to_num(lags.select(target_columns).to_numpy().astype(np.float32), nan=0.0, posinf=5.0, neginf=-5.0)
    target = np.zeros_like(cache["target"])
    target_mask = np.zeros_like(cache["target_mask"])

    for row_index, (time_id, symbol_id) in enumerate(zip(time_values, symbol_values)):
        time_slot = METADATA.time_to_slot.get(int(time_id))
        if time_slot is None:
            continue
        insertion = np.searchsorted(METADATA.symbol_ids, int(symbol_id))
        known = insertion < len(METADATA.symbol_ids) and METADATA.symbol_ids[insertion] == symbol_id
        slot = int(insertion) if known else cache["slot_by_unknown_symbol"].get(int(symbol_id))
        if slot is None or not cache["valid"][time_slot, slot]:
            continue
        target[time_slot, slot] = target_values[row_index]
        target_mask[time_slot, slot] = True

    if not target_mask.any():
        return None
    batch = {name: value for name, value in cache.items() if name in {"features", "tokens", "valid", "weights"}}
    batch["target"] = target
    batch["target_mask"] = target_mask
    return to_jax_batch(batch)


class JaneStreetRuntime:
    def __init__(self, parameter_stack: dict[str, Any]):
        self.params = parameter_stack
        model_shape = jax.tree_util.tree_leaves(parameter_stack)[0].shape[:2]
        self.n_models = int(np.prod(model_shape))
        self.current_date: int | None = None
        self.current_day: dict[str, Any] | None = None
        self.carries = self._fresh_carries()

    def _fresh_carries(self) -> tuple[jax.Array, ...]:
        return tuple(
            jnp.zeros((N_LOCAL_DEVICES, MODELS_PER_DEVICE, MODEL_CFG.max_symbols, MODEL_CFG.gru_hidden), dtype=jnp.float32)
            for _ in range(MODEL_CFG.num_blocks)
        )

    def _adapt_from_lags(self, lags: pl.DataFrame | None) -> None:
        if self.current_day is None:
            return
        batch = batch_with_lagged_targets(lags, self.current_day)
        if batch is None:
            return
        self.params, loss = PMAP_ONLINE_UPDATE(self.params, replicated_batch(batch))
        print(f"online adaptation loss: {np.asarray(loss).mean():.6f}")

    def predict(self, test: pl.DataFrame, lags: pl.DataFrame | None) -> pl.DataFrame:
        date_id = int(test["date_id"][0])
        if self.current_date != date_id:
            if self.current_date is not None:
                self._adapt_from_lags(lags)
            self.current_date = date_id
            self.current_day = empty_runtime_day()
            self.carries = self._fresh_carries()

        step, slots, _ = prepare_inference_step(test, self.current_day)
        prediction, self.carries = PMAP_PREDICT_STEP(self.params, replicated_batch(step), self.carries)
        responder_6 = np.clip(np.asarray(prediction)[0, slots, 6], -5.0, 5.0)
        return pl.DataFrame({"row_id": test["row_id"], "responder_6": responder_6})


RUNTIME = JaneStreetRuntime(PARAMETER_SHARDS)


def predict(test: pl.DataFrame, lags: pl.DataFrame | None) -> pl.DataFrame:
    return RUNTIME.predict(test, lags)

## Submit or validate locally

Kaggle's evaluation package invokes `predict` timestep by timestep. The local
gateway path is only for the public mock data; Kaggle uses `serve()` on a
competition rerun.

In [ ]:
from kaggle_evaluation.jane_street_inference_server import JSInferenceServer

# The local Kaggle template passes `file_share_dir`, while this competition's
# JSInferenceServer hook uses the older one-argument signature.
if not hasattr(JSInferenceServer, "_original_get_gateway_for_test"):
    JSInferenceServer._original_get_gateway_for_test = JSInferenceServer._get_gateway_for_test

    def _compatible_get_gateway_for_test(self, data_paths=None, file_share_dir=None, *args, **kwargs):
        return self._original_get_gateway_for_test(data_paths)

    JSInferenceServer._get_gateway_for_test = _compatible_get_gateway_for_test

if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    inference_server = JSInferenceServer(predict)
    inference_server.serve()
else:
    # Collect local gateway prediction batches and write a submission CSV.
    local_predictions = []

    def predict_and_collect(test: pl.DataFrame, lags: pl.DataFrame | None) -> pl.DataFrame:
        result = predict(test, lags)
        local_predictions.append(result)
        return result

    # JSInferenceServer routes requests by callback name; the gateway calls `predict`.
    predict_and_collect.__name__ = "predict"
    inference_server = JSInferenceServer(predict_and_collect)
    inference_server.run_local_gateway(
        (str(DATA_ROOT / "test.parquet"), str(DATA_ROOT / "lags.parquet"))
    )
    submission = pl.concat(local_predictions)
    submission_path = Path("submission.csv")
    submission.write_csv(submission_path)
    print(f"Saved {submission.height} predictions to {submission_path.resolve()}")